In [1]:
import pandas as pd
import numpy as np
import csv
import hazm
import re
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
import pingouin as pg  

C:\Users\armin\AppData\Local\Temp\ipykernel_7616\3854743248.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


# Reading file

In [ ]:

# خواندن فایل‌های ارزیابی انسانی و مدل
human = pd.read_csv('human file.csv') # Human sentences
model = pd.read_csv('model file.csv') # Model sentences

# Lexical Idea

In [ ]:
from transformers import AutoTokenizer, BertModel
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import normalize
import torch
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("HooshvareLab/bert-base-parsbert-uncased")
model = BertModel.from_pretrained("HooshvareLab/bert-base-parsbert-uncased")

def robust_idea_count(text, eps=0.4):

    if not isinstance(text, str) or not text.strip():
        return 0

    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)

    with torch.no_grad():
        outputs = model(**inputs)

    hidden = outputs.last_hidden_state[0]  # [seq_len, hidden]

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    # remove special tokens
    valid_idx = [
        i for i, t in enumerate(tokens)
        if t not in ["[CLS]", "[SEP]", "[PAD]"]
    ]

    tokens = [tokens[i] for i in valid_idx]
    embeddings = hidden[valid_idx].cpu().numpy()

    if len(embeddings) == 0:
        return 0

    embeddings = normalize(embeddings)

    db = DBSCAN(
        eps=eps,
        min_samples=2,   
        metric="cosine"
    ).fit(embeddings)

    labels = db.labels_

    # حذف noise (-1)
    unique_clusters = set(labels) - {-1}

    return len(unique_clusters)


In [ ]:
results = []

# Or model
for text in human['Sentence']:   # dataset = list of 100 textshuman
    out = robust_idea_count(text)
    results.append(out)

print(results)

# Human

In [ ]:
human_average = pd.read_csv('file.csv') # Human average score
fluency_human = human_average['Fluency']


# Model

In [ ]:
model_average = pd.read_csv('file_model.csv') # Model average score
fluency_model = model_average['Fluency']

# Get Correlation

In [ ]:
from scipy.stats import spearmanr

rho, p = spearmanr(x, y)

print("Spearman correlation:", rho)
print("p-value:", p)


# Save File

In [ ]:
lexical_idea_human.to_csv("file.csv", index=False)

In [ ]:
lexical_idea_model.to_csv("file.csv", index=False)